# Conv2DTranspose, and the checkerboard artifact

How upsampling with a learned kernel works, why it produces a grid pattern, and the two ways to avoid it.

**Runs on:** CPU — about 2 minutes &nbsp;·&nbsp; **Slides:** [Chapter 11 — Image Segmentation](../../../course-web-slides/ch11/index.html) &nbsp;·&nbsp; **Section:** 02 — Upsampling

---

## What it does to a shape

In [ ]:
import keras
from keras import layers
import numpy as np

x = np.zeros((1, 8, 8, 4), dtype="float32")

for stride in [1, 2, 4]:
    out = layers.Conv2DTranspose(4, 3, strides=stride, padding="same")(x)
    print(f"strides={stride}: {x.shape} -> {out.shape}")

Expected output:

```
strides=1: (1, 8, 8, 4) -> (1, 8, 8, 4)
strides=2: (1, 8, 8, 4) -> (1, 16, 16, 4)
strides=4: (1, 8, 8, 4) -> (1, 32, 32, 4)
```

The inverse of a strided convolution in **shape** — not in value. It is sometimes called a *deconvolution*, which is wrong and misleading: nothing is being deconvolved.

## The mechanism

In [ ]:
import matplotlib.pyplot as plt

# One hot pixel, one fixed kernel, so the mechanism is visible.
inp = np.zeros((1, 4, 4, 1), dtype="float32")
inp[0, 1, 1, 0] = 1.0
inp[0, 2, 3, 0] = 1.0

ct = layers.Conv2DTranspose(1, 3, strides=2, padding="same",
                            use_bias=False,
                            kernel_initializer="ones")
out = ct(inp).numpy()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(8, 3.6))
a1.imshow(inp[0, :, :, 0], cmap="gray_r"); a1.set_title("input 4x4, two hot pixels")
a2.imshow(out[0, :, :, 0], cmap="gray_r"); a2.set_title("output 8x8")
for a in (a1, a2): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

Each input pixel is **multiplied by the whole kernel** and stamped into the output at its strided position. Where the stamps overlap, the contributions add — and that overlap is where the artifact comes from.

## The checkerboard

In [ ]:
# kernel_size=3 with strides=2 gives uneven overlap.
bad = layers.Conv2DTranspose(1, 3, strides=2, padding="same",
                             use_bias=False, kernel_initializer="ones")
ones = np.ones((1, 16, 16, 1), dtype="float32")
out_bad = bad(ones).numpy()[0, :, :, 0]

# kernel_size divisible by stride overlaps evenly.
good = layers.Conv2DTranspose(1, 4, strides=2, padding="same",
                              use_bias=False, kernel_initializer="ones")
out_good = good(ones).numpy()[0, :, :, 0]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
a1.imshow(out_bad, cmap="gray"); a1.set_title(
    f"kernel 3, stride 2 -- values {np.unique(out_bad)}")
a2.imshow(out_good, cmap="gray"); a2.set_title(
    f"kernel 4, stride 2 -- values {np.unique(out_good)}")
for a in (a1, a2): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

A uniform input produces a **striped output**. With kernel 3 and stride 2, some output pixels receive two contributions and some receive one — the grid pattern is baked into the arithmetic, before any learning happens.

## Two fixes

In [ ]:
# Fix 1: make kernel_size divisible by strides.
fix1 = keras.Sequential([layers.Conv2DTranspose(32, 4, strides=2,
                                                padding="same")])

# Fix 2: separate the upsampling from the convolution entirely.
fix2 = keras.Sequential([
    layers.UpSampling2D(size=2, interpolation="bilinear"),
    layers.Conv2D(32, 3, padding="same"),
])

probe = np.ones((1, 16, 16, 8), dtype="float32")
print("Conv2DTranspose(4, stride 2):", fix1(probe).shape,
      f"{fix1.count_params():,} params")
print("UpSampling + Conv2D:        ", fix2(probe).shape,
      f"{fix2.count_params():,} params")

Fix 2 is what chapter 17's U-Net uses, and it is the one to reach for by default. **Interpolate, then convolve** — the upsampling is fixed and artifact-free, and the convolution learns what to do with the result.

## Seeing it in a real decoder

In [ ]:
def decoder(kind):
    keras.utils.set_random_seed(0)
    i = keras.Input(shape=(8, 8, 64))
    x = i
    for f in [64, 32, 16]:
        if kind == "transpose":
            x = layers.Conv2DTranspose(f, 3, strides=2, padding="same",
                                       activation="relu")(x)
        else:
            x = layers.UpSampling2D(2, interpolation="bilinear")(x)
            x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
    o = layers.Conv2D(1, 3, padding="same", activation="sigmoid")(x)
    return keras.Model(i, o)

probe = np.random.default_rng(0).normal(size=(1, 8, 8, 64)).astype("float32")
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, kind in zip(axes, ["transpose", "upsample"]):
    out = decoder(kind)(probe).numpy()[0, :, :, 0]
    ax.imshow(out, cmap="gray"); ax.set_title(kind); ax.axis("off")
plt.suptitle("Untrained decoders on the same input", y=1.0)
plt.tight_layout(); plt.show()

Untrained, so this shows the **structural bias** rather than anything learned. Training reduces the artifact; it does not remove it, and generative models in chapter 17 are where it becomes most visible.

---

## What to take away

- `Conv2DTranspose` stamps a learned kernel at strided positions and adds the overlaps.
- Uneven overlap produces the **checkerboard artifact**, before any training.
- Fix by making kernel size divisible by stride, or by separating upsampling from convolution.
- `UpSampling2D` + `Conv2D` is the safer default, and what chapter 17's U-Net uses.